# Erzeugte Wahlkreis-JSON-Dateien prüfen

Dieses Notebook liest nur die fertigen Dateien ein. Die fachliche Herleitung und der Vergleich mit den amtlichen Rändern stehen bereits Schritt für Schritt im Aufbereitungsnotebook. Hier geht es um Dateigröße, Struktur und offensichtliche Datenfehler vor der Übernahme in die App.

In [ ]:
from pathlib import Path
import json
import math

import pandas as pd
from IPython.display import display


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("Das Notebook muss innerhalb des Repository-Ordners laufen.")


ROOT = find_repository_root()
GENERATED_DIRECTORY = ROOT / "scripts/data/generated"
FIRST_VOTES_JSON = GENERATED_DIRECTORY / "first_votes.json"
SECOND_VOTES_JSON = GENERATED_DIRECTORY / "second_votes.json"

## 1. Dateien und Größen

In [ ]:
for path in (FIRST_VOTES_JSON, SECOND_VOTES_JSON):
    if not path.is_file():
        raise FileNotFoundError(f"Datei fehlt: {path}")

file_sizes = pd.DataFrame(
    {
        "file": [FIRST_VOTES_JSON.name, SECOND_VOTES_JSON.name],
        "MiB": [
            FIRST_VOTES_JSON.stat().st_size / 1024**2,
            SECOND_VOTES_JSON.stat().st_size / 1024**2,
        ],
    }
)
display(file_sizes)
print(f"Zusammen: {file_sizes['MiB'].sum():.2f} MiB")

## 2. JSON lesen und erste Zeilen ansehen

In [ ]:
first_votes = pd.read_json(FIRST_VOTES_JSON)
second_votes = pd.read_json(SECOND_VOTES_JSON)

print(f"Erststimmen-Zeilen: {len(first_votes):,}")
print(f"Zweitstimmen-Zeilen: {len(second_votes):,}")
display(first_votes.head(20))
display(second_votes.head(20))

## 3. Erwartete Spalten und Wertebereiche

In [ ]:
expected_columns = {
    "districtId",
    "state",
    "gender",
    "ageGroup",
    "party",
    "voteType",
    "electionMethod",
    "votes",
}

for label, frame, expected_vote_type in (
    ("first_votes", first_votes, 1),
    ("second_votes", second_votes, 2),
):
    missing = expected_columns - set(frame.columns)
    unexpected = set(frame.columns) - expected_columns
    print(label, {"missing": sorted(missing), "unexpected": sorted(unexpected)})
    assert not missing
    assert not unexpected
    assert set(frame["voteType"].astype(str)) == {str(expected_vote_type)}
    assert set(frame["gender"]) <= {"m", "w"}
    assert set(frame["ageGroup"]) <= {"18-24", "25-34", "35-44", "45-54", "55-64", "65+"}
    assert set(frame["electionMethod"]) <= {"postal", "in-person"}
    assert frame["districtId"].notna().all()
    assert (frame["districtId"] > 0).all()
    assert frame["votes"].notna().all()
    assert frame["votes"].map(math.isfinite).all()
    assert (frame["votes"] >= 0).all()

print("Grundlegende Wertebereiche sind gültig.")

## 4. Doppelte Detailzeilen suchen

In [ ]:
key_columns = [
    "districtId",
    "state",
    "gender",
    "ageGroup",
    "party",
    "voteType",
    "electionMethod",
]

for label, frame in (("first_votes", first_votes), ("second_votes", second_votes)):
    duplicates = frame[frame.duplicated(key_columns, keep=False)]
    print(f"{label}: {len(duplicates):,} doppelte Zeilen")
    display(duplicates.head(20))
    assert duplicates.empty

## 5. Abdeckung ansehen

In [ ]:
coverage = pd.DataFrame(
    {
        "first_votes": [
            first_votes["districtId"].nunique(),
            first_votes["state"].nunique(),
            first_votes["party"].nunique(),
        ],
        "second_votes": [
            second_votes["districtId"].nunique(),
            second_votes["state"].nunique(),
            second_votes["party"].nunique(),
        ],
    },
    index=["Wahlkreise", "Bundesländer", "Parteien/Sammelkategorien"],
)
display(coverage)

## 6. Beispiel eines Wahlkreises

In [ ]:
sample_district = int(first_votes["districtId"].min())

display(
    first_votes[first_votes["districtId"] == sample_district]
    .groupby(["party", "electionMethod"], as_index=False)["votes"]
    .sum()
    .sort_values("votes", ascending=False)
    .head(30)
)